# Diabetes Disease-Progression Regression

**Goal:** predict a quantitative measure of diabetes disease progression one year
after baseline, using the classic scikit-learn diabetes dataset.

A compact but complete **regression** workflow: EDA, cross-validated model
comparison, tuning, and proper evaluation (R2 / MAE / RMSE).

**Dataset:** 442 patients, 10 baseline features (already mean-centered and
scaled): age, sex, BMI, blood pressure, and six blood-serum measurements.

## 1. Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
data = load_diabetes(as_frame=True)
X = data.data
y = data.target
print(X.shape)
X.head()

## 2. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y, bins=30, color="#4c72b0")
axes[0].set_title("Target distribution (disease progression)")
axes[0].set_xlabel("Progression score")

corr_with_target = X.apply(lambda col: np.corrcoef(col, y)[0, 1]).sort_values()
sns.barplot(x=corr_with_target.values, y=corr_with_target.index, ax=axes[1], color="#55a868")
axes[1].set_title("Feature correlation with target")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix")
plt.show()

## 3. Model comparison (5-fold cross-validation)

The features are already standardized, but we keep a `StandardScaler` in the
pipeline so the workflow stays correct for the linear models if reused on raw data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

candidates = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

rows = []
for name, est in candidates.items():
    pipe = Pipeline([("scale", StandardScaler()), ("model", est)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2", n_jobs=-1)
    rows.append({"model": name, "cv_r2": scores.mean(), "cv_std": scores.std()})
    print(f"{name:<20} R2 {scores.mean():.4f} +/- {scores.std():.4f}")

cv_results = pd.DataFrame(rows).sort_values("cv_r2", ascending=False).reset_index(drop=True)
cv_results

## 4. Tune Gradient Boosting

In [ ]:
gb_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
])
param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [1, 2, 3],
    "model__subsample": [0.7, 0.85, 1.0],
}
search = RandomizedSearchCV(gb_pipe, param_dist, n_iter=25, cv=5,
                            scoring="r2", random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
search.fit(X_train, y_train)
print("Best CV R2:", round(search.best_score_, 4))
for k, v in search.best_params_.items():
    print(f"  {k} = {v}")

## 5. Evaluation on the held-out test set

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print("Test R2  :", round(r2_score(y_test, y_pred), 4))
print("Test MAE :", round(mean_absolute_error(y_test, y_pred), 4))
print("Test RMSE:", round(root_mean_squared_error(y_test, y_pred), 4))

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.6, color="#4c72b0")
lims = [y.min(), y.max()]
plt.plot(lims, lims, "r--", lw=2)
plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.title("Predicted vs actual")
plt.tight_layout(); plt.show()

## 6. Feature importance

In [ ]:
importances = best_model.named_steps["model"].feature_importances_
importance_df = (pd.DataFrame({"feature": X.columns, "importance": importances})
                 .sort_values("importance", ascending=False).reset_index(drop=True))
plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x="importance", y="feature", color="#4c72b0")
plt.title("Feature importance (tuned Gradient Boosting)")
plt.tight_layout(); plt.show()
importance_df

## 7. Summary

- Benchmarked linear (Linear/Ridge/Lasso) and ensemble models with 5-fold CV.
- This dataset has a real performance ceiling (~0.45-0.50 R2 is typical); honest
  reporting matters more than chasing an inflated number.
- BMI and serum measure **s5** are consistently the strongest predictors.

**Possible extensions:** polynomial/interaction features for the linear models,
and partial-dependence plots for BMI and blood pressure.